## Neural Network Model:

A multi-output neural network is defined using TensorFlow/Keras.
The input is based on 5 key soil features (NPK, water holding capacity, and pH).
Shared layers are used across the network to extract common patterns from the input data, followed by separate layers for each output.

The network has 7 outputs, each solving a different task:
1. Fertilizer Type: Multiclass classification (4 categories)
2. Application Rate: Regression
3. Timing of Application: Regression
4. Organic vs. Synthetic: Binary classification
5. pH Adjustment: Multiclass classification (4 categories)
6. Soil Testing Frequency: Regression
7. Organic Matter Improvement: Regression


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split

This code first generates synthetic data.

In [ ]:
# Function to generate synthetic data
def generate_synthetic_data(num_samples):
    # Random NPK levels (in mg/kg)
    npk_levels = np.random.randint(0, 200, size=(num_samples, 3))

    # Random water holding capacity (in %)
    water_holding_capacity = np.random.uniform(10, 60, size=(num_samples, 1))

    # Random pH (0-14 scale)
    ph_levels = np.random.uniform(4, 9, size=(num_samples, 1))

    # Fertilizer type (categorical: 0=Balanced, 1=N-rich, 2=P-rich, 3=K-rich)
    fertilizer_type = np.random.randint(0, 4, size=(num_samples, 1))

    # Application rate (in kg/ha)
    application_rate = np.random.uniform(50, 300, size=(num_samples, 1))

    # Timing of application (in days)
    timing_application = np.random.uniform(0, 180, size=(num_samples, 1))

    # Organic vs Synthetic Fertilizer (binary: 0=Organic, 1=Synthetic)
    organic_vs_synthetic = np.random.randint(0, 2, size=(num_samples, 1))

    # Soil pH adjustment recommendation (categorical: 0=None, 1=Lime, 2=Sulfur, 3=Gypsum)
    ph_adjustment = np.random.randint(0, 4, size=(num_samples, 1))

    # Soil testing frequency (in months)
    soil_testing = np.random.uniform(1, 12, size=(num_samples, 1))

    # Organic matter improvement (in tons/ha)
    organic_matter_improvement = np.random.uniform(0.5, 5, size=(num_samples, 1))

    # Combine features and outputs into a dataframe
    data = np.hstack((npk_levels, water_holding_capacity, ph_levels, fertilizer_type,
                      application_rate, timing_application, organic_vs_synthetic,
                      ph_adjustment, soil_testing, organic_matter_improvement))

    # print(data[:5])

    # Return features (inputs) and labels (outputs)
    X = data[:, :5]  # Inputs: NPK, water holding capacity, pH
    y = {
        'fertilizer_type': data[:, 5],
        'application_rate': data[:, 6],
        'timing_application': data[:, 7],
        'organic_vs_synthetic': data[:, 8],
        'ph_adjustment': data[:, 9],
        'soil_testing': data[:, 10],
        'organic_matter_improvement': data[:, 11]
    }
    return X, y

# Generate the synthetic dataset
num_samples = 1000
X, y = generate_synthetic_data(num_samples)


[[154.         193.         119.          45.84749109   7.33863947
    1.         156.28450294  61.52910374   0.           3.
    6.07888984   3.47288699]
 [ 89.          27.         156.          48.37436093   4.55584178
    2.         154.07595274  71.51740341   1.           3.
    6.62063821   1.170123  ]
 [122.         133.         108.          30.5558063    6.26500547
    0.         251.64736927  87.08491213   0.           0.
   11.78918121   2.77793339]
 [184.          84.         181.          35.29184525   8.88322425
    0.         254.6156425  150.35698674   0.           3.
    1.79708296   2.45956365]
 [148.         199.          40.          18.24678148   4.52373549
    0.          84.39322755 150.77904161   1.           3.
    5.46801631   2.67399146]]


In [ ]:
# Split the data into training and testing sets for the input (X)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Split the data for each output (y) individually
y_train = {}
y_test = {}
for key in y.keys():
    y_train[key], y_test[key] = train_test_split(y[key], test_size=0.2, random_state=42)

# Convert outputs to list format, ensuring proper shapes for model fitting
y_train_list = [y_train[key] if key in ['fertilizer_type', 'ph_adjustment'] else y_train[key].reshape(-1, 1) for key in y.keys()]
y_test_list = [y_test[key] if key in ['fertilizer_type', 'ph_adjustment'] else y_test[key].reshape(-1, 1) for key in y.keys()]


In [ ]:
# Model definition
inputs = Input(shape=(5,))  # 5 features: NPK levels, water holding capacity, pH

# Shared layers
x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)

# Output 1: Fertilizer Type and Composition
fertilizer_type = Dense(64, activation='relu')(x)
fertilizer_type_output = Dense(4, activation='softmax', name='fertilizer_type')(fertilizer_type)

# Output 2: Application Rates
application_rate = Dense(32, activation='relu')(x)
application_rate_output = Dense(1, activation='linear', name='application_rate')(application_rate)

# Output 3: Timing of Application
timing_application = Dense(32, activation='relu')(x)
timing_application_output = Dense(1, activation='linear', name='timing_application')(timing_application)

# Output 4: Organic vs Synthetic Fertilizer
organic_vs_synthetic = Dense(32, activation='relu')(x)
organic_vs_synthetic_output = Dense(1, activation='sigmoid', name='organic_vs_synthetic')(organic_vs_synthetic)

# Output 5: pH Adjustment (Lime, Sulfur, etc.)
ph_adjustment = Dense(32, activation='relu')(x)
ph_adjustment_output = Dense(4, activation='softmax', name='ph_adjustment')(ph_adjustment)

# Output 6: Soil Testing and Monitoring
soil_testing = Dense(32, activation='relu')(x)
soil_testing_output = Dense(1, activation='linear', name='soil_testing')(soil_testing)

# Output 7: Organic Matter Improvement
organic_matter_improvement = Dense(32, activation='relu')(x)
organic_matter_output = Dense(1, activation='linear', name='organic_matter_improvement')(organic_matter_improvement)

# Combine the model outputs
model = Model(inputs=inputs, outputs=[fertilizer_type_output, application_rate_output,
                                      timing_application_output, organic_vs_synthetic_output,
                                      ph_adjustment_output, soil_testing_output,
                                      organic_matter_output])

In [ ]:
# Compile the model with metrics for each output
model.compile(optimizer='adam',
              loss={
                  'fertilizer_type': 'sparse_categorical_crossentropy',  # for classification
                  'application_rate': 'mse',  # for regression
                  'timing_application': 'mse',  # for regression
                  'organic_vs_synthetic': 'binary_crossentropy',  # for binary classification
                  'ph_adjustment': 'sparse_categorical_crossentropy',  # for classification
                  'soil_testing': 'mse',  # for regression
                  'organic_matter_improvement': 'mse'  # for regression
              },
              metrics={
                  'fertilizer_type': ['accuracy'],  # classification accuracy
                  'application_rate': ['mse'],  # mean squared error
                  'timing_application': ['mse'],  # mean squared error
                  'organic_vs_synthetic': ['accuracy'],  # binary classification accuracy
                  'ph_adjustment': ['accuracy'],  # classification accuracy
                  'soil_testing': ['mse'],  # mean squared error
                  'organic_matter_improvement': ['mse']  # mean squared error
              })

In [ ]:
# Train the model
history = model.fit(X_train, y_train_list, epochs=50, batch_size=32, validation_data=(X_test, y_test_list))


Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - application_rate_mse: 20430.5996 - fertilizer_type_accuracy: 0.2464 - loss: 27319.7324 - organic_matter_improvement_mse: 29.5455 - organic_vs_synthetic_accuracy: 0.4802 - ph_adjustment_accuracy: 0.2417 - soil_testing_mse: 33.6133 - timing_application_mse: 6801.3877 - val_application_rate_mse: 8526.8936 - val_fertilizer_type_accuracy: 0.1900 - val_loss: 12595.8076 - val_organic_matter_improvement_mse: 5.8120 - val_organic_vs_synthetic_accuracy: 0.5100 - val_ph_adjustment_accuracy: 0.2700 - val_soil_testing_mse: 13.3199 - val_timing_application_mse: 4036.5850
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - application_rate_mse: 8592.8281 - fertilizer_type_accuracy: 0.2384 - loss: 12061.9395 - organic_matter_improvement_mse: 5.3488 - organic_vs_synthetic_accuracy: 0.5144 - ph_adjustment_accuracy: 0.2492 - soil_testing_mse: 14.8692 - timing_application_mse: 3438.7019 - val_application_rate_mse: 7138.0293 - val_fertilizer_type_accurac

In [ ]:
# Evaluate the model on test data
loss, *metrics = model.evaluate(X_test, y_test_list)
print(f"Test Loss: {loss}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - application_rate_mse: 5347.2158 - fertilizer_type_accuracy: 0.2264 - loss: 7984.8057 - organic_matter_improvement_mse: 2.4287 - organic_vs_synthetic_accuracy: 0.5483 - ph_adjustment_accuracy: 0.2906 - soil_testing_mse: 10.3817 - timing_application_mse: 2620.9570 
Test Loss: 8595.849609375
